# Personal Chef Agent (Gemini)

An agent that suggests recipes from your leftover ingredients, using web search.

In [ ]:
%pip install -q langchain langchain-google-genai langgraph tavily-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [ ]:
import os

# Get a Google key at aistudio.google.com and a Tavily key at app.tavily.com
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6LJacP_8AQcxUKyB-Hg03jG2RTDC0BJNRT5uqQHlhjjZQ"
os.environ["TAVILY_API_KEY"] = "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"

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)

In [ ]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': "Leftover cooked chicken and rice are fantastic building blocks for a quick, delicious meal! Since both are already cooked, your prep time will be minimal. \n\nHere are a few great recipe ideas you can make, depending on what else you have in your fridge and pantry:\n\n---\n\n### 1. **Classic Chicken Fried Rice** *(10–15 Minutes)*\n* **Why it works:** Cold, leftover rice actually works *better* than freshly cooked rice for fried rice because it doesn't get mushy.\n* **Key ingredients:** Leftover chicken & rice, eggs, soy sauce, sesame oil (or regular oil), garlic, green onions, and frozen veggies (peas & carrots).\n* **Vibe:** Quick, savory, pan-fried weeknight favorite.\n\n---\n\n### 2. **Cheesy Chicken & Rice Casserole** *(30–35 Minutes)*\n* **Why it works:** Baking leftovers together in a sauce revives the moisture and melds the flavors.\n* **Key ingredients:** Leftover chicken & rice, cream of mushroom/chicken soup (or melted cheese sauce), broccoli or mix

In [ ]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='16794c81-c1ab-474f-92c2-be4a8fd04a61'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "leftover chicken and rice recipes"}'}, '__gemini_function_call_thought_signatures__': {'call_398754': 'EogGCoUGAWkUfROpyS5M2cf1ruKemHUKpFG7gB9W3TZFL/0El+qJ0ZmCh05kV4EsxEyWQyKVTXYcfbqUT03tQzqnlyClxDLSInEt2PeLlksJR5G1w4IKHNBuKoi7HpB5hQIDTyvyVym/sN6EWlSRUnVgc0+fTZ5SQD1Nch0PHmeFQ5bQpiso3Xl1y79/XWN7Y6f5cST5DWE6XDu/UJGp9UYEr2l7UP4a07rVHMy+zDyKKSMccLPquYu9mW9xJQnMF+8vAuiT+0yUZkeqVS1E0DhvYEMvH+ZBTng3Z5QZ6O6qC1TpKVoM1aJYg+pbl2JCbJE0T5JbnVZjDC2yOCFloYbTG1kWYd2qiZohfyZAXqN1WMvJm+Zplj06uhZk/DwRSn7Rg51Mz0fX/TyhkjAXU8VR1d2MVtd1d1eQRLj6C21y9w2TYh/W+ucQpUWGABkUyYOQp4zlLbTGsPCPaPNvyCBRT3fV7i+cMo3JEu3SZtMwhk50tjXiaYdtN4Vh9X6oaIsL3wSlY5DdMh5gB+627Wuc+43shgpEtbbAG4FfR+KQaMEQgI5lhXOjMgJLGHENFt3d7H60LrvKCuzTq9B

## Follow-up (same thread, so the agent remembers the conversation)

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Give me the full instructions for the first recipe.")]},
    config
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'Here is the step-by-step recipe for **Classic Chicken Fried Rice**. Because your chicken and rice are already cooked, this meal comes together in just about 10 to 15 minutes!\n\n---\n\n### **Prep & Cook Time**\n* **Prep time:** 5 minutes\n* **Cook time:** 10 minutes\n* **Servings:** 2–3\n\n---\n\n### **Ingredients**\n* **3 cups** leftover cold cooked rice *(break up any large clumps with your hands before cooking)*\n* **1 to 1½ cups** leftover cooked chicken, diced or shredded\n* **2 large** eggs, lightly whisked\n* **2 tbsp** cooking oil (vegetable, canola, or peanut oil works best)\n* **1 tsp** sesame oil *(optional, but adds great flavor)*\n* **2 cloves** garlic, minced (or ½ tsp garlic powder)\n* **1 cup** frozen peas and carrots mix (no need to thaw)\n* **3–4 tbsp** soy sauce (adjust to taste; low-sodium preferred)\n* **2–3** green onions (scallions), thinly sliced\n\n---\n\n### **Instructions**\n\n#### **Step 1: Prep your rice and eggs**\nUse your finge